# Melodía

**Trabajo Práctico — Parseo y Generación de código**  
Alumno: Matías Angeli

---

## Objetivo

Melodía es un lenguaje de dominio específico de temática musical. Un programa describe una canción: se le da un nombre y, dentro de un bloque, se escriben sentencias que definen el tempo, agregan notas con su duración, arman acordes, intercalan silencios y repiten fragmentos. Como salida informa la duración total de la canción.

## Alcance

Es un lenguaje declarativo y secuencial. Permite definir una canción con nombre y un conjunto de sentencias:

- **Tempo**: la velocidad en pulsos por minuto (bpm).
- **Nota**: una nota con su duración.
- **Acorde**: varias notas que suenan juntas.
- **Silencio**: una pausa con su duración.
- **Repetir**: repite un bloque de sentencias una cantidad de veces (puede anidarse).

Las notas válidas son do, re, mi, fa, sol, la, si, y las duraciones redonda, blanca, negra, corchea y semicorchea. El lenguaje verifica que todo esté bien formado y calcula la duración total. Quedan fuera del alcance las variables, las expresiones aritméticas y las condiciones.

## Especificaciones léxicas

| Categoría | ER | Describe… |
|-----------|----|-----------|
| Palabras reservadas | `(Cancion\|Tempo\|Nota\|Acorde\|Silencio\|Repetir)` | Construcciones del lenguaje |
| Identificador (nota o duración) | `(a\|...\|z)+` | Palabra en minúscula; si es una nota o duración válida se decide en la parte semántica |
| Entero | `(0\|...\|9)+` | Tempo y cantidad de repeticiones |
| Cadena | `"(.)*"` | Nombre de la canción |
| Agrupadores | `(\{\|\})` | Abren y cierran bloques |
| Delimitador de sentencia | `;` | Marca el fin de una sentencia |
| Separadores | `( \|\t\|\n)+` | Espacios, tabs y saltos de línea; se ignoran |

## Especificaciones sintácticas

```
grammar Melodia;

cancion : 'Cancion' STRING '{' sentencia+ '}' ;

sentencia
    : 'Tempo' INT ';'
    | 'Nota' nota duracion ';'
    | 'Acorde' nota+ ';'
    | 'Silencio' duracion ';'
    | 'Repetir' INT '{' sentencia+ '}' ';'
    ;

nota     : ID ;
duracion : ID ;

ID     : [a-z]+ ;
INT    : [0-9]+ ;
STRING : '"' .*? '"' ;
WS     : [ \t\r\n]+ -> skip ;
```

## Especificaciones semánticas

**Palabras reservadas**

| Palabra reservada | Para qué sirve |
|-------------------|----------------|
| Cancion | Define la canción y su nombre. |
| Tempo | Fija la velocidad en bpm. |
| Nota | Agrega una nota con su duración. |
| Acorde | Varias notas que suenan juntas. |
| Silencio | Pausa con su duración. |
| Repetir | Repite un bloque una cantidad de veces. |

---
**Qué verifica**

| Regla | Qué hace |
|-------|----------|
| Nota válida | Cada nota debe ser una de: do, re, mi, fa, sol, la, si. |
| Duración válida | Cada duración debe ser: redonda, blanca, negra, corchea o semicorchea. |
| Tempo único | El Tempo puede definirse a lo sumo una vez y debe ser mayor a 0. Si falta, se asume 100 bpm. |
| Repetir ≥ 1 | La cantidad de repeticiones debe ser al menos 1. |
| Acorde ≥ 2 notas | Un acorde necesita por lo menos dos notas. |
| Duración total | Suma la duración (en negras); Repetir multiplica. Con el tempo calcula los segundos. |

---
**Tipos de datos**

No tiene tipos ni variables (es declarativo: solo describe la canción). Los controles son los de la tabla anterior.

---
**Los tres tipos de error**

| Etapa | Ejemplo |
|-------|---------|
| Léxico | Un caracter que no corresponde, como `Nota do @;`. |
| Sintáctico | Falta un `;` o un `}`, por ejemplo `Nota do negra` (sin `;`). |
| Semántico | Una nota o duración inexistente, por ejemplo `Nota do larga;`. |

## Scanner, Parser y Acciones semánticas

El código completo está en el archivo `melodia.py`: el scanner (`ply.lex`), el parser (`ply.yacc`, LALR(1)) y la clase `Interprete` con las acciones semánticas. En la celda siguiente lo importamos para usarlo.

In [ ]:
from melodia import (analizar, reproducir, EJEMPLO, ErrorLexico, ErrorSintactico, ErrorSemantico)

## Ejemplo

In [11]:
print(EJEMPLO)
reproducir(EJEMPLO)

Cancion "Escala alegre" {
    Tempo 120;
    Nota do negra;
    Nota re negra;
    Nota mi blanca;
    Acorde do mi sol;
    Silencio negra;
    Repetir 2 {
        Nota sol corchea;
        Nota la corchea;
    };
}
Cancion "Escala alegre": 8 negras, 4.00 s a 120 bpm


{'nombre': 'Escala alegre', 'beats': 8.0, 'tempo': 120, 'segundos': 4.0}

## Casos de prueba automatizados

In [12]:
import unittest

# Los casos de prueba están en test_melodia.py
suite = unittest.defaultTestLoader.loadTestsFromName('test_melodia')
unittest.TextTestRunner(verbosity=2).run(suite)

test_caracter_invalido (test_melodia.Lexicos.test_caracter_invalido) ... ok
test_acorde_una_nota (test_melodia.Semanticos.test_acorde_una_nota) ... ok
test_duracion_invalida (test_melodia.Semanticos.test_duracion_invalida) ... ok
test_nota_invalida (test_melodia.Semanticos.test_nota_invalida) ... ok
test_repetir_cero (test_melodia.Semanticos.test_repetir_cero) ... ok
test_tempo_duplicado (test_melodia.Semanticos.test_tempo_duplicado) ... ok
test_tempo_no_positivo (test_melodia.Semanticos.test_tempo_no_positivo) ... ok
test_cancion_vacia (test_melodia.Sintacticos.test_cancion_vacia) ... ok
test_falta_llave (test_melodia.Sintacticos.test_falta_llave) ... ok
test_falta_punto_y_coma (test_melodia.Sintacticos.test_falta_punto_y_coma) ... ok
test_duraciones (test_melodia.Validas.test_duraciones) ... ok
test_ejemplo_completo (test_melodia.Validas.test_ejemplo_completo) ... ok
test_repetir_anidado (test_melodia.Validas.test_repetir_anidado) ... ok
test_tempo_por_defecto (test_melodia.Validas.t

<unittest.runner.TextTestResult run=14 errors=0 failures=0>